# NeoNatal Watch AI — Phase 7: Autoencoder Anomaly Detection

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook builds an unsupervised deep learning model. Instead of explicitly teaching the model what a 'deterioration event' looks like, we only teach it what 'normal' looks like.

---

## The Theory:
- The **Encoder** compresses 30 minutes of 6 vital signs down into a tiny bottleneck array.
- The **Decoder** tries to decompress that bottleneck back into the original 30-minute window.
- Because we ONLY train it on healthy patients, it becomes very good at compressing/decompressing normal heart rates and oxygen levels.
- If a patient suddenly drops in oxygen (Deterioration), the model has never seen this pattern before. It will fail to decompress it accurately. 
- We measure this failure as the **Mean Squared Error (MSE)**. High MSE = High Risk of Deterioration!

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from ml.models.autoencoder import build_autoencoder_model
from ml.evaluation.metrics import evaluate_model

print('Imports OK')

In [ ]:
# Load 3D tensors
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

# ISOLATE NORMAL DATA
X_train_normal = X_train[y_train == 0]
print(f"Original Train Shape: {X_train.shape}")
print(f"Normal Train Shape:   {X_train_normal.shape}")

In [ ]:
# Build the Autoencoder
model = build_autoencoder_model(
    sequence_length=X_train.shape[1],
    n_features=X_train.shape[2]
)

model.summary()

In [ ]:
# Train the Model
# Notice that X is both the input AND the target!
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

history = model.fit(
    X_train_normal, X_train_normal, # Unsupervised
    validation_split=0.15,
    epochs=5,
    batch_size=64,
    callbacks=callbacks
)

In [ ]:
# Visualize Reconstruction Error (Anomaly Score)
X_test_pred = model.predict(X_test)
mse = np.mean(np.square(X_test - X_test_pred), axis=(1, 2))

# Plot the distribution of errors for Normal vs Event
plt.figure(figsize=(10, 6))
plt.hist(mse[y_test==0], bins=50, alpha=0.6, color='steelblue', label='Normal (Label 0)', density=True)
plt.hist(mse[y_test==1], bins=50, alpha=0.6, color='crimson', label='Event (Label 1)', density=True)
plt.title('Autoencoder Reconstruction Error Distribution')
plt.xlabel('Mean Squared Error (Anomaly Score)')
plt.ylabel('Density')
plt.legend()
plt.show()